# 3. Statystyczny ML

TF-IDF, n-gramy oraz SVM, Naive Bayes i Random Forest — lokalne uczenie i predykcja.

## Protokół wspólny

Dane lokalne: pełny `combined_reduced.json` (77 677 rekordów, `1` = phishing,
`0` = benign); mieszanina URL, SMS, e-maili i HTML, a nie tylko e-maile.
Każdy notatnik sprawdza SHA-256 wszystkich części i oryginału.
Po odrzuceniu pustych wejść, konfliktów etykiet i duplikatów znormalizowanych
pierwszych 4000 znaków wybieramy warstwowo **8000 rekordów** z ziarnem 42.
Używamy tego samego podziału train/validation/test = **60/20/20**.
To demonstracja na próbce; pełny zbiór pozostaje w `data`.

Wszystkie reprezentacje uczymy tylko na train. Walidacja służy do wyboru progu
w metodach nadzorowanych; test zostaje na końcowy pomiar. Zbiory są rozłączne
po kluczu normalizacji, lecz bliskie duplikaty lub rodziny kampanii nadal mogą
przeciekać: brak dat i identyfikatorów kampanii uniemożliwia wiarygodny test czasowy.
Nie odwiedzamy linków ani nie renderujemy HTML. Nie korzystamy z API modeli,
internetu podczas wykonania ani pobieranych modeli językowych.

Czasy to pomiar CPU w tej maszynie, z mediany 3 wywołań po rozgrzewce, w ms/próbkę
w batchu testowym (nie opóźnienie pojedynczego żądania). Obejmują transformację tekstu,
ale nie odczyt plików. Rozmiar to nieskompresowana serializacja pickle obiektu
potrzebnego do predykcji, bez środowiska Pythona i danych. Wartości między rodzinami
zadań nie są automatycznie porównywalne. Czułość = recall klasy phishing;
FPR = FP / (FP + TN). Average precision podsumowuje ranking precision–recall.

In [1]:
import sys
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Uruchomienie z katalogu repozytorium albo z notebooks/.
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'notebooks' / 'common.py').exists())
sys.path.insert(0, str(ROOT / 'notebooks'))
from common import (SEED, load_splits, classification_metrics, choose_threshold,
                    timed_prediction, size_mib, save_results)
train, validation, test = load_splits()
y_train = train.label.to_numpy()
y_val = validation.label.to_numpy()
y_test = test.label.to_numpy()
x_train = train.input.tolist()
x_val = validation.input.tolist()
x_test = test.input.tolist()
display(pd.DataFrame({'train': train.label.value_counts(),
                      'validation': validation.label.value_counts(),
                      'test': test.label.value_counts()}).rename_axis('label'))

{
  "source_records": 77677,
  "deduplicated_sample": 8000,
  "conflicting_keys_removed": 3,
  "seed": 42,
  "train": 4800,
  "validation": 1600,
  "test": 1600,
  "max_input_chars": 4000,
  "split_sha256": {
    "train": "dbd41bff6349f412f1cbc777e2a7e19e75f23d09586914fd0518a620cd0caf18",
    "validation": "5ef4b8c140c03e6f97c2a792e2887a785aac5d59986fd08ea41784e87d85904b",
    "test": "ce29c3a3e537a8f6d97ad38fd43fe2c5952ff6ed04c7611991a05c8784e0aa5c"
  }
}


,train,validation,test
label,,,
0,2802,934,934
1,1998,666,666


## Plan doświadczenia: reprezentacja × klasyfikator

Dziewięć konfiguracji: trzy reprezentacje × trzy klasyfikatory.

- słowa (1,1): prosty TF-IDF;
- słowa (1,2): dodajemy frazy dwuwyrazowe;
- znaki (3,5): fragmenty adresów URL, literówki i warianty zapisu.

Każdy wektoryzator: max_features=20 000, min_df=2, sublinear_tf=True.
LinearSVC (liniowy SVM): C=1, class_weight=balanced, max_iter=5000.
MultinomialNB: alpha=1. RandomForest: 120 drzew, max_depth=30,
min_samples_leaf=2, class_weight=balanced_subsample, n_jobs=2.
Są to stałe ustawienia dydaktyczne, bez przeszukiwania hiperparametrów na teście.

Dla każdej konfiguracji wybieramy próg maksymalizujący F1 na validation.
SVM daje margines, a NB/RF prawdopodobieństwo; nie porównujemy ich bezpośrednio
ani nie uznajemy za skalibrowane prawdopodobieństwa. Precision i recall mierzymy
przy wybranym progu, ROC-AUC/AP na ciągłych wynikach. Testu nie używamy do wyboru modelu.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from sklearn.metrics import f1_score

representations = {
    'word 1-gram': dict(analyzer='word', ngram_range=(1, 1)),
    'word 1-2-gram': dict(analyzer='word', ngram_range=(1, 2)),
    'char 3-5-gram': dict(analyzer='char', ngram_range=(3, 5)),
}
classifiers = {
    'SVM': LinearSVC(C=1, class_weight='balanced', max_iter=5000, random_state=SEED),
    'Naive Bayes': MultinomialNB(alpha=1),
    'Random Forest': RandomForestClassifier(n_estimators=120, max_depth=30,
                        min_samples_leaf=2, class_weight='balanced_subsample',
                        random_state=SEED, n_jobs=2),
}
rows = []
for representation, params in representations.items():
    # Wspólna reprezentacja w obrębie trzech klasyfikatorów; fit wyłącznie train.
    vectorizer = TfidfVectorizer(**params, max_features=20000, min_df=2,
                                 sublinear_tf=True, dtype=np.float32)
    start = perf_counter()
    X = vectorizer.fit_transform(x_train)
    vectorizer_fit_s = perf_counter() - start
    for name, estimator in classifiers.items():
        model = clone(estimator)
        start = perf_counter()
        model.fit(X, y_train)
        fit_s = perf_counter() - start + vectorizer_fit_s
        pipeline = Pipeline([('tfidf', vectorizer), ('classifier', model)])
        def predict_scores(texts, p=pipeline, is_svm=(name == 'SVM')):
            return p.decision_function(texts) if is_svm else p.predict_proba(texts)[:, 1]
        val_scores = predict_scores(x_val)
        threshold = choose_threshold(y_val, val_scores)
        val_f1 = f1_score(
            y_val, val_scores >= threshold, zero_division=0)
        scores, latency = timed_prediction(predict_scores, x_test)
        rows.append(dict(method=name, representation=representation, threshold=threshold,
                         validation_f1=val_f1, fit_s=fit_s, ms_per_sample=latency,
                         model_mib=size_mib((pipeline, threshold)), features=X.shape[1],
                         **classification_metrics(y_test, scores >= threshold, scores)))
        print(representation, '/', name, 'gotowe')
assert len(rows) == 9

word 1-gram / SVM gotowe


word 1-gram / Naive Bayes gotowe


word 1-gram / Random Forest gotowe


word 1-2-gram / SVM gotowe


word 1-2-gram / Naive Bayes gotowe


word 1-2-gram / Random Forest gotowe


char 3-5-gram / SVM gotowe


char 3-5-gram / Naive Bayes gotowe


char 3-5-gram / Random Forest gotowe


## Łączne podsumowanie porównawcze

Wszystkie dziewięć konfiguracji na identycznym teście. `fit_s` zawiera uczenie
wektoryzatora i klasyfikatora, ale nie wybór progu. `model_mib` obejmuje kompletną
ścieżkę predykcji: słownik, IDF, klasyfikator i próg. Wyniki czasu zawierają TF-IDF.
Wybór rekomendacji poniżej odbywa się na **validation_f1**, nie na testowym F1.

In [3]:
summary = save_results('03_statystyczny_ml', rows)
display(summary.sort_values('validation_f1', ascending=False).round(4))
selected = summary.loc[summary.validation_f1.idxmax()]
display(Markdown(f"Najwyższe F1 walidacyjne: **{selected['representation']} + {selected['method']}** "
                 f"({selected.validation_f1:.3f}). Na nietkniętym teście: "
                 f"F1 **{selected.f1:.3f}**, czułość **{selected.recall:.3f}**, "
                 f"FPR **{selected.fpr:.3f}**; "
                 f"**{selected.ms_per_sample:.3f} ms/próbkę**, "
                 f"**{selected.model_mib:.2f} MiB**. "
                 "Ocena rekomendacji nadal wymaga niezależnego testu czasowego."))
# Średnie po reprezentacjach są tylko opisem tych trzech wariantów, nie CV.
display(summary.groupby('method')[['recall', 'precision', 'f1', 'ms_per_sample', 'model_mib']]
        .mean().round(4))

,method,representation,threshold,validation_f1,fit_s,ms_per_sample,model_mib,features,accuracy,precision,recall,f1,fpr,roc_auc,average_precision,tn,fp,fn,tp
6,SVM,char 3-5-gram,-0.0839,0.8981,12.1478,1.2451,0.7264,20000,0.9106,0.8818,0.9069,0.8942,0.0867,0.9687,0.9562,853,81,62,604
8,Random Forest,char 3-5-gram,0.4738,0.8714,15.2087,1.0924,5.0672,20000,0.8900,0.8561,0.8844,0.8700,0.1060,0.9516,0.9281,835,99,77,589
0,SVM,word 1-gram,-0.0485,0.8681,0.3608,0.0582,0.6521,16607,0.8812,0.8315,0.8964,0.8627,0.1296,0.9540,0.9370,813,121,69,597
3,SVM,word 1-2-gram,-0.1872,0.8675,1.5968,0.1107,0.8268,20000,0.8675,0.7866,0.9354,0.8546,0.1809,0.9534,0.9367,765,169,43,623
7,Naive Bayes,char 3-5-gram,0.3521,0.8514,11.6180,1.1531,1.1842,20000,0.8481,0.7878,0.8694,0.8266,0.1670,0.9256,0.8966,778,156,87,579
4,Naive Bayes,word 1-2-gram,0.3175,0.8410,1.5466,0.1160,1.2845,20000,0.8362,0.7506,0.9084,0.8220,0.2152,0.9298,0.9105,733,201,61,605
1,Naive Bayes,word 1-gram,0.3177,0.8240,0.3252,0.0632,1.0322,16607,0.8188,0.7547,0.8363,0.7934,0.1938,0.9153,0.8977,753,181,109,557
2,Random Forest,word 1-gram,0.5231,0.8095,0.9134,0.0815,3.9893,16607,0.8212,0.7589,0.8363,0.7957,0.1895,0.9041,0.8888,757,177,109,557
5,Random Forest,word 1-2-gram,0.5235,0.8079,2.2396,0.1525,3.9040,20000,0.8156,0.7419,0.8544,0.7941,0.2120,0.9044,0.8868,736,198,97,569


Najwyższe F1 walidacyjne: **char 3-5-gram + SVM** (0.898). Na nietkniętym teście: F1 **0.894**, czułość **0.907**, FPR **0.087**; **1.245 ms/próbkę**, **0.73 MiB**. Ocena rekomendacji nadal wymaga niezależnego testu czasowego.

,recall,precision,f1,ms_per_sample,model_mib
method,,,,,
Naive Bayes,0.8714,0.7644,0.8140,0.4441,1.1669
Random Forest,0.8584,0.7856,0.8200,0.4421,4.3201
SVM,0.9129,0.8333,0.8705,0.4713,0.7351


**Wnioski:** Naive Bayes ma prosty model i szybkie uczenie, ale zakłada warunkową
niezależność cech i może źle kalibrować pewność. Liniowy SVM pasuje do rzadkich
wielowymiarowych tekstów; margines wymaga kalibracji przed interpretacją jako
prawdopodobieństwo. Random Forest wychwytuje nieliniowe zależności, lecz dla
rzadkich tekstów może być większy i wolniejszy. N-gramy słowne zachowują krótkie
frazy, a znakowe obsługują warianty URL kosztem transformacji tekstu.

Zestawienie pokazuje kompromis jakości, czułości, FPR, czasu i rozmiaru na tej
próbce, a nie uniwersalną przewagę jednej metody. Limit 4000 znaków może usuwać
końcową część maila. W produkcji potrzebne są testy kampanii/domen niewidzianych
w treningu, analiza driftu i progi wynikające z kosztu błędów. Te modele nie
rozumieją intencji tak jak człowiek i nie gwarantują odporności na GenAI/Zero-Day.